# E-commerce Recommendation System

## Data Science Capstone Project

**Dataset:** E-commerce Behavior Data from Multi-Category Store

**Analysis Period:** October 2019

**Project Date:** August 2026

**Author:** Said Uzun

## Objective

This notebook develops a recommendation system using the user-product interaction dataset created in the previous stage of the project.

The main objectives are:

- Prepare the interaction data for recommendation modeling.
- Build a recommendation model based on implicit user feedback.
- Generate personalized product recommendations.
- Demonstrate example recommendations for selected users.

## Import Libraries

The required Python libraries are imported for data preparation and recommendation modeling.

In [ ]:
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors

> **Note:** This notebook requires the `user_product_interactions.csv` file generated in Notebook 2.  
> If the Colab runtime is restarted, the file must be uploaded again before running the recommendation pipeline.

In [ ]:
interactions = pd.read_csv("user_product_interactions.csv")

interactions.head()

In [ ]:
print("Rows:", len(interactions))
print("Unique users:", interactions["user_id"].nunique())
print("Unique products:", interactions["product_id"].nunique())

In [ ]:
interactions.describe()

In [ ]:
interactions["interaction_score"].describe()

The interaction dataset contains over 23 million user-product relationships generated from implicit feedback. Each interaction is represented by a weighted interaction score based on user behavior.

## Building the User–Product Interaction Matrix

Recommendation algorithms require the interaction data to be represented as a matrix where each row corresponds to a user, each column corresponds to a product, and each cell contains the interaction score between them.

Since the dataset is extremely sparse, a sparse matrix representation is used to reduce memory usage and improve computational efficiency.

In [ ]:
user_codes = interactions["user_id"].astype("category")
product_codes = interactions["product_id"].astype("category")

In [ ]:
interaction_matrix = csr_matrix(
    (
        interactions["interaction_score"],
        (
            user_codes.cat.codes,
            product_codes.cat.codes
        )
    )
)

interaction_matrix

In [ ]:
print("Matrix shape:", interaction_matrix.shape)
print("Non-zero interactions:", interaction_matrix.nnz)

## Normalizing User Vectors

Before computing similarities, each user's interaction vector is normalized.

This prevents highly active users from dominating similarity calculations simply because they interacted with many products. The recommendation model will focus on behavioral patterns rather than interaction volume.

In [ ]:
normalized_matrix = normalize(interaction_matrix)

In [ ]:
normalized_matrix

Each row now has unit length, allowing cosine similarity to compare users based on interaction patterns instead of absolute interaction counts.

## Item-Based Collaborative Filtering

To make the recommendation process computationally efficient, the model uses item-based collaborative filtering.

Products are considered similar when they receive similar interaction patterns from users. Cosine similarity is used to measure the similarity between product interaction vectors.

In [ ]:
product_support = (
    interactions
    .groupby("product_id")["user_id"]
    .nunique()
    .sort_values(ascending=False)
)

top_products = product_support.head(5000).index

model_interactions = interactions[
    interactions["product_id"].isin(top_products)
].copy()

print("Model rows:", len(model_interactions))
print("Model users:", model_interactions["user_id"].nunique())
print("Model products:", model_interactions["product_id"].nunique())

## Training the Recommendation Model

A K-Nearest Neighbors (KNN) model with cosine distance is used to identify similar products based on user interaction patterns.

The model does not require explicit ratings and therefore is well suited for implicit feedback recommendation systems.

In [ ]:
item_codes = model_interactions["product_id"].astype("category")
user_codes = model_interactions["user_id"].astype("category")

item_user_matrix = csr_matrix(
    (
        model_interactions["interaction_score"],
        (
            item_codes.cat.codes,
            user_codes.cat.codes
        )
    )
)

item_user_matrix

In [ ]:
print("Item matrix shape:", item_user_matrix.shape)

In [ ]:
knn_model = NearestNeighbors(
    metric="cosine",
    algorithm="brute",
    n_neighbors=6
)

knn_model.fit(item_user_matrix)

## Similar Product Recommendations

The trained model is used to identify products with similar user interaction patterns.  
Cosine distance is converted into a similarity score, where values closer to 1 indicate stronger similarity.

In [ ]:
product_ids = item_codes.cat.categories.to_numpy()

product_to_index = pd.Series(
    np.arange(len(product_ids)),
    index=product_ids
)

In [ ]:
def get_similar_products(product_id, n=5):

    if product_id not in product_to_index.index:
        return pd.DataFrame()

    product_index = int(product_to_index.loc[product_id])

    distances, indices = knn_model.kneighbors(
        item_user_matrix[product_index],
        n_neighbors=n + 1
    )

    similar_indices = indices.flatten()[1:]
    similarities = 1 - distances.flatten()[1:]

    return pd.DataFrame({
        "product_id": product_ids[similar_indices],
        "similarity_score": similarities
    })

In [ ]:
example_product = top_products[0]

print("Target Product:", example_product)

similar_products = get_similar_products(
    example_product,
    n=5
)

similar_products

## Personalized Product Recommendations

Personalized recommendations are generated by combining the products a user has previously interacted with and the most similar products identified by the item-based collaborative filtering model.

Products already seen by the user are excluded from the final recommendation list.

In [ ]:
def recommend_for_user(user_id, n=5, neighbors_per_item=10):

    user_history = model_interactions[
        model_interactions["user_id"] == user_id
    ]

    if user_history.empty:
        return pd.DataFrame()

    seen_products = set(user_history["product_id"])

    recommendation_scores = {}

    for _, row in user_history.iterrows():

        product_id = row["product_id"]
        interaction_score = row["interaction_score"]

        if product_id not in product_to_index.index:
            continue

        product_index = int(product_to_index.loc[product_id])

        distances, indices = knn_model.kneighbors(
            item_user_matrix[product_index],
            n_neighbors=neighbors_per_item + 1
        )

        for distance, neighbor_index in zip(
            distances.flatten()[1:],
            indices.flatten()[1:]
        ):

            recommended_product = product_ids[neighbor_index]

            if recommended_product in seen_products:
                continue

            similarity = 1 - distance

            recommendation_scores[recommended_product] = (
                recommendation_scores.get(recommended_product, 0)
                + similarity * interaction_score
            )

    recommendations = pd.DataFrame(
        recommendation_scores.items(),
        columns=["product_id", "recommendation_score"]
    )

    return (
        recommendations
        .sort_values("recommendation_score", ascending=False)
        .head(n)
        .reset_index(drop=True)
    )

In [ ]:
user_activity = (
    model_interactions
    .groupby("user_id")["product_id"]
    .nunique()
)

candidate_users = user_activity[
    (user_activity >= 10) &
    (user_activity <= 30)
]

example_user = candidate_users.index[0]

print("Example User:", example_user)
print("Products Interacted:", candidate_users.loc[example_user])

In [ ]:
user_recommendations = recommend_for_user(
    example_user,
    n=5
)

user_recommendations.round(3)

The recommendation model successfully generated personalized product suggestions for the selected user.

The recommended products were not previously interacted with by the user and were ranked according to the weighted similarity scores derived from item-based collaborative filtering.

## Conclusion

This notebook developed an item-based collaborative filtering recommendation system using implicit user feedback.

The recommendation pipeline included:

- building a sparse user-product interaction matrix,
- training a k-Nearest Neighbors model using cosine similarity,
- identifying similar products,
- generating personalized product recommendations based on weighted similarity scores.

The resulting recommendation engine demonstrates how behavioral interaction data can be transformed into meaningful product suggestions without requiring explicit user ratings.